# External Framework Validation — SynthEval

**Clean, linear pipeline.** Run top to bottom (Kernel → Restart & Run All).

This notebook re-computes the SynthEval fidelity and privacy metrics for
CTGAN and CTAB-GAN+ on the UCI Default of Credit Card Clients dataset.

Every reused variable is defined exactly once, in order, so that a full
"Run All" is guaranteed reproducible. Assertions guard the critical steps.

**Requirements:** environment `tese_eval` (Python 3.10) with `syntheval`, pandas, numpy, scikit-learn installed.


## 0. Configuration and paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from syntheval import SynthEval

RANDOM_STATE = 123


DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)

Data folder: data


## 1. Canonical column definitions

These are defined **once** here and never reassigned. `FEATURES` is the 23
predictor columns, deliberately **excluding** the target `def_pay`.

In [2]:
TARGET   = "def_pay"

FEATURES = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
            "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
            "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
            "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

MONETARY = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] \
                         + [f"PAY_AMT{i}"  for i in range(1, 7)]

# Categorical/ordinal columns (matches the manual TVD treatment)
CAT_COLS = ["SEX", "EDUCATION", "MARRIAGE"] + [f"PAY_{i}" for i in range(1, 7)]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 23, f"FEATURES must contain 23, but has {len(FEATURES)}"
assert TARGET not in FEATURES, "def_pay cannot be included in FEATURES"
assert len(CAT_COLS) == 9 and len(NUM_COLS) == 14

print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))

FEATURES: 23 | CAT: 9 | NUM: 14


## 2. Helper: align any dataframe to the 23 features

Drops `def_pay` if present, enforces column order, casts monetary columns to
`float64` (so CTAB-GAN+ integer output matches the rest).

In [3]:
def to_features(df, name):
    d = df.copy()
    if TARGET in d.columns:
        d = d.drop(columns=[TARGET])
    d = d[FEATURES]                       # enforce order; errors if any missing
    d[MONETARY] = d[MONETARY].astype("float64")
    assert list(d.columns) == FEATURES
    print(f"{name:<16} -> {d.shape}")
    return d

## 3. Load the synthetic datasets and the real minority reference

In [4]:
real_min  = pd.read_csv(DATA_DIR / "real_minority_train.csv")
synth_ctg = pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv")
synth_ctb = pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv")

real_min_p  = to_features(real_min,  "real_minority")
synth_ctg_p = to_features(synth_ctg, "ctgan")
synth_ctb_p = to_features(synth_ctb, "ctabganplus")

# all three must share identical columns
for d in (synth_ctg_p, synth_ctb_p):
    assert list(d.columns) == list(real_min_p.columns)
print("\nAll columns aligned")

real_minority    -> (4645, 23)
ctgan            -> (11710, 23)
ctabganplus      -> (11710, 23)

All columns aligned


## 4. Reconstruct the canonical split → holdout

Rebuilds the 70/30 stratified split from raw data with `RANDOM_STATE = 123`,
then verifies it matches `real_train_full.csv` exactly. The minority records of the
test partition (1,991 rows) become the privacy holdout, matching the
minority-only reference used by the generators and the other frameworks.

In [5]:
df_raw = pd.read_csv(DATA_DIR / "UCI_Credit_Card.csv")

df_c = (df_raw
        .drop(columns=["ID"])
        .rename(columns={"PAY_0": "PAY_1",
                         "default.payment.next.month": TARGET}))
df_c["EDUCATION"] = df_c["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df_c["MARRIAGE"]  = df_c["MARRIAGE"].replace({0: 3})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
assert train_recon.equals(real_full.reset_index(drop=True)), \
    "Reconstructed split != real_train_full.csv"
print("Split reproduced bit for bit. OK.")

# Privacy holdout = the MINORITY records of the test set, matching the
# minority-only training reference (as in the Bank notebook)
holdout = to_features(X_test[y_test == 1], "holdout")
assert list(holdout.columns) == FEATURES

Split reproduced bit for bit. OK.
holdout          -> (1991, 23)


## 5. Fidelity metrics (no holdout needed)

TVD, KS, correlation difference and pMSE for the two generators.

In [6]:
ev_fid = SynthEval(real_min_p, cat_cols=CAT_COLS)

def fidelity(synth, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    return ev_fid.evaluate(synth, ks_test={}, corr_diff={}, p_mse={})

res_ctgan = fidelity(synth_ctg_p,   "CTGAN")
res_ctab  = fidelity(synth_ctb_p,   "CTAB-GAN+")

Rich console is not supported in this environment. Defaulting to ascii console.

CTGAN
SynthEval: synthetic data read successfully


Syntheval: p_mse: 100%|██████████| 3/3 [00:42<00:00, 14.08s/it]    


SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| General Kolmogorov–Smirnov Statistic       :  0.1656   0.0118  |
|   -> Avg. Kolmogorov–Smirnov dist.         :  0.1675   0.0178  |
|   -> Avg. Total Variation Distance         :  0.1627   0.0131  |
| Fraction of Significant KS Tests           :  1.0000           |
|   -> # of Significant Tests at a=0.05      :  23.0000           |
|   -> Avg. combined p-value                 :  0.0004   0.0001  |
| Mixed correlation matrix difference        :  3.6542           |
| Propensity mean squared error (pMSE)       :  0.0248   0.0005  |
|   -> average pMSE classifier accuracy      :  0.5572   0.0033  |
+----------------------------------------------------------------+


CTAB-GAN+
SynthEval: synthetic data read successfully


Syntheval: p_mse: 100%|██████████| 3/3 [00:34<00:00, 11.49s/it]    


SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| General Kolmogorov–Smirnov Statistic       :  0.1814   0.0170  |
|   -> Avg. Kolmogorov–Smirnov dist.         :  0.2261   0.0181  |
|   -> Avg. Total Variation Distance         :  0.1117   0.0144  |
| Fraction of Significant KS Tests           :  1.0000           |
|   -> # of Significant Tests at a=0.05      :  23.0000           |
|   -> Avg. combined p-value                 :  0.0012   0.0008  |
| Mixed correlation matrix difference        :  1.5405           |
| Propensity mean squared error (pMSE)       :  0.0098   0.0002  |
|   -> average pMSE classifier accuracy      :  0.4440   0.0019  |
+----------------------------------------------------------------+



In [7]:
def tag(res, model):
    d = res.copy(); d.insert(0, "model", model); return d

fidelity_all = pd.concat([tag(res_ctgan, "CTGAN"),
                          tag(res_ctab,  "CTAB-GAN+")], ignore_index=True)

print(fidelity_all.pivot(index="model", columns="metric", values="val").round(4))
fidelity_all.to_csv(DATA_DIR / "syntheval_fidelity_results.csv", index=False)
print("\nSaved: syntheval_fidelity_results.csv")

metric     avg_pMSE  corr_mat_diff  frac_ks_sigs  ks_tvd_stat
model                                                        
CTAB-GAN+    0.0098         1.5405           1.0       0.1814
CTGAN        0.0248         3.6542           1.0       0.1656

Saved: syntheval_fidelity_results.csv


## 6. Privacy metrics — Variant A (PAY_X categorical)

Uses the holdout. NNAA, DCR and epsilon risk. This is the slow step
(~2-3 min per model due to eps_risk).

In [8]:
ev_priv_A = SynthEval(real_min_p, holdout_dataframe=holdout, cat_cols=CAT_COLS)

def privacy(evaluator, synth, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    return evaluator.evaluate(synth, nnaa={}, dcr={}, eps_risk={})

privA_ctgan = privacy(ev_priv_A, synth_ctg_p,   "CTGAN (A)")
privA_ctab  = privacy(ev_priv_A, synth_ctb_p,   "CTAB-GAN+ (A)")

Rich console is not supported in this environment. Defaulting to ascii console.

CTGAN (A)
SynthEval: synthetic data read successfully


Syntheval: eps_risk: 100%|██████████| 3/3 [02:23<00:00, 47.77s/it]


SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| Nearest neighbour adversarial accuracy     :  0.7681   0.0007  |
+----------------------------------------------------------------+

Privacy metric description                      value    error
+----------------------------------------------------------------+
| Privacy loss (diff. in NNAA)               : -0.0344   0.0012  |
| Median distance to closest record          :  4.8669           |
| Epsilon identifiability risk               :  0.0988           |
| Privacy loss (diff. in eps. risk)          :  0.0146           |
+----------------------------------------------------------------+


CTAB-GAN+ (A)
SynthEval: synthetic data read successfully


Syntheval: eps_risk: 100%|██████████| 3/3 [02:25<00:00, 48.39s/it]

SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| Nearest neighbour adversarial accuracy     :  0.6141   0.0006  |
+----------------------------------------------------------------+

Privacy metric description                      value    error
+----------------------------------------------------------------+
| Privacy loss (diff. in NNAA)               : -0.0177   0.0016  |
| Median distance to closest record          :  1.6480           |
| Epsilon identifiability risk               :  0.2319           |
| Privacy loss (diff. in eps. risk)          :  0.0555           |
+----------------------------------------------------------------+



## 7. Privacy metrics — Variant B (PAY_X numerical)

Same as above but PAY_X treated as numerical, for the DCR sensitivity study.

In [9]:
CAT_COLS_B = ["SEX", "EDUCATION", "MARRIAGE"]   # PAY_X now numerical

ev_priv_B = SynthEval(real_min_p, holdout_dataframe=holdout, cat_cols=CAT_COLS_B)

privB_ctgan = privacy(ev_priv_B, synth_ctg_p,   "CTGAN (B)")
privB_ctab  = privacy(ev_priv_B, synth_ctb_p,   "CTAB-GAN+ (B)")

Rich console is not supported in this environment. Defaulting to ascii console.

CTGAN (B)
SynthEval: synthetic data read successfully


Syntheval: eps_risk: 100%|██████████| 3/3 [01:40<00:00, 33.52s/it]


SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| Nearest neighbour adversarial accuracy     :  0.7762   0.0006  |
+----------------------------------------------------------------+

Privacy metric description                      value    error
+----------------------------------------------------------------+
| Privacy loss (diff. in NNAA)               : -0.0364   0.0013  |
| Median distance to closest record          :  2.8661           |
| Epsilon identifiability risk               :  0.0876           |
| Privacy loss (diff. in eps. risk)          :  0.0189           |
+----------------------------------------------------------------+


CTAB-GAN+ (B)
SynthEval: synthetic data read successfully


Syntheval: eps_risk: 100%|██████████| 3/3 [01:40<00:00, 33.43s/it]

SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| Nearest neighbour adversarial accuracy     :  0.6133   0.0005  |
+----------------------------------------------------------------+

Privacy metric description                      value    error
+----------------------------------------------------------------+
| Privacy loss (diff. in NNAA)               : -0.0196   0.0017  |
| Median distance to closest record          :  1.4793           |
| Epsilon identifiability risk               :  0.2299           |
| Privacy loss (diff. in eps. risk)          :  0.0609           |
+----------------------------------------------------------------+



## 8. Consolidate privacy results and DCR sensitivity

In [10]:
def tagv(res, model, variant):
    d = res.copy()
    d.insert(0, "variant", variant)
    d.insert(0, "model", model)
    return d

privacy_all = pd.concat([
    tagv(privA_ctgan, "CTGAN",     "A_PAY_categorical"),
    tagv(privA_ctab,  "CTAB-GAN+", "A_PAY_categorical"),
    tagv(privB_ctgan, "CTGAN",     "B_PAY_numerical"),
    tagv(privB_ctab,  "CTAB-GAN+", "B_PAY_numerical"),
], ignore_index=True)

privacy_all.to_csv(DATA_DIR / "syntheval_privacy_results.csv", index=False)

for v in ["A_PAY_categorical", "B_PAY_numerical"]:
    sub = privacy_all[privacy_all["variant"] == v]
    print(f"\n=== {v} ===")
    print(sub.pivot(index="model", columns="metric", values="val").round(4))


=== A_PAY_categorical ===
metric     eps_identif_risk  median_DCR    nnaa  priv_loss_eps  priv_loss_nnaa
model                                                                         
CTAB-GAN+            0.2319      1.6480  0.6141         0.0555         -0.0177
CTGAN                0.0988      4.8669  0.7681         0.0146         -0.0344

=== B_PAY_numerical ===
metric     eps_identif_risk  median_DCR    nnaa  priv_loss_eps  priv_loss_nnaa
model                                                                         
CTAB-GAN+            0.2299      1.4793  0.6133         0.0609         -0.0196
CTGAN                0.0876      2.8661  0.7762         0.0189         -0.0364


In [11]:
# DCR sensitivity table
print("metric names:", privacy_all["metric"].unique())

dcr = privacy_all[privacy_all["metric"] == "median_DCR"]
sens = dcr.pivot(index="model", columns="variant", values="val")
sens.columns = ["PAY_categorical", "PAY_numerical"]
sens["delta_%"] = ((sens["PAY_numerical"] / sens["PAY_categorical"]) - 1) * 100

print("\nDCR SENSITIVITY TO TYPE DECLARATION")
print(sens.round(4))
sens.to_csv(DATA_DIR / "syntheval_dcr_sensitivity.csv")
print("\nSaved: syntheval_dcr_sensitivity.csv")

metric names: ['nnaa' 'priv_loss_nnaa' 'median_DCR' 'eps_identif_risk' 'priv_loss_eps']

DCR SENSITIVITY TO TYPE DECLARATION
           PAY_categorical  PAY_numerical  delta_%
model                                             
CTAB-GAN+           1.6480         1.4793 -10.2368
CTGAN               4.8669         2.8661 -41.1101

Saved: syntheval_dcr_sensitivity.csv


---
**Done.** If every cell ran without error, all three result CSVs are written
and the in-memory variables (`real_min_p`, `holdout`, `synth_*_p`, results)
are available for any further analysis.

In [12]:
print("=== FIDELITY ===")
print(fidelity_all.pivot(index="model", columns="metric", values="val").round(4))

print("\n=== DCR SENSITIVITY ===")
print(sens.round(4))

=== FIDELITY ===
metric     avg_pMSE  corr_mat_diff  frac_ks_sigs  ks_tvd_stat
model                                                        
CTAB-GAN+    0.0098         1.5405           1.0       0.1814
CTGAN        0.0248         3.6542           1.0       0.1656

=== DCR SENSITIVITY ===
           PAY_categorical  PAY_numerical  delta_%
model                                             
CTAB-GAN+           1.6480         1.4793 -10.2368
CTGAN               4.8669         2.8661 -41.1101
